# 🩺 Step 1: Data Cleaning, Audit & Clinical Synthetic Augmentation
This notebook cleans, audits, and enriches three clinical datasets:
1. **Heart Disease Dataset**
2. **Parkinson's Disease Dataset**
3. **Diabetes Dataset**

To overcome small sample sizes (especially Parkinson's $N=195$ and Heart Disease $N=302$), we apply **Class-Conditional Statistical Data Augmentation**:
- Preserves covariance and correlation matrices $\mathcal{N}(\mu_c, \Sigma_c)$
- Preserves discrete feature probability distributions
- Clips values within realistic physiological boundaries
- Combines original clinical records with synthetic samples into robust, balanced mixed datasets.

In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "Dataset minor"
OUTPUT_DIR = "processed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Clean Original Raw Datasets

In [2]:
# --- 1.1 Heart Disease (Deduplicate to unique clinical cases) ---
raw_heart = pd.read_csv(os.path.join(DATA_DIR, "heart.csv")).drop_duplicates()
print(f"Heart Disease Unique Cleaned: {raw_heart.shape[0]} rows, {raw_heart.shape[1]} columns")

# --- 1.2 Parkinson's Disease (Drop non-clinical 'name' column) ---
raw_park = pd.read_csv(os.path.join(DATA_DIR, "parkinsons.data"))
if 'name' in raw_park.columns:
    raw_park = raw_park.drop(columns=['name'])
print(f"Parkinson's Cleaned: {raw_park.shape[0]} rows, {raw_park.shape[1]} columns")

# --- 1.3 Diabetes (Class-stratified median imputation for biological zeros) ---
diab_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
             'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
raw_diab = pd.read_csv(os.path.join(DATA_DIR, "pima-indians-diabetes.csv"), names=diab_cols)

zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
raw_diab[zero_cols] = raw_diab[zero_cols].replace(0, np.nan)
for col in zero_cols:
    median_by_class = raw_diab.groupby('Outcome')[col].transform('median')
    raw_diab[col] = raw_diab[col].fillna(median_by_class)

print(f"Diabetes Cleaned & Imputed: {raw_diab.shape[0]} rows, {raw_diab.shape[1]} columns")

Heart Disease Unique Cleaned: 302 rows, 14 columns
Parkinson's Cleaned: 195 rows, 23 columns
Diabetes Cleaned & Imputed: 768 rows, 9 columns


## 2. Statistical Synthetic Data Generator
Generates realistic clinical samples conditioned on disease status, preserving covariance, correlations, and physiological bounds.

In [3]:
def generate_synthetic_data(df, target_col, target_total_samples=2000, random_state=42):
    np.random.seed(random_state)
    classes = df[target_col].unique()
    n_per_class = target_total_samples // len(classes)
    
    synthetic_records = []
    
    for c in classes:
        sub_df = df[df[target_col] == c].drop(columns=[target_col])
        n_needed = n_per_class - len(sub_df)
        if n_needed <= 0:
            continue
            
        continuous_cols = []
        discrete_cols = []
        for col in sub_df.columns:
            if sub_df[col].nunique() <= 5 and sub_df[col].dtype in ['int64', 'float64'] and (sub_df[col] % 1 == 0).all():
                discrete_cols.append(col)
            else:
                continuous_cols.append(col)
                
        # Continuous features: Class-conditional Multivariate Normal
        if continuous_cols:
            mean_vec = sub_df[continuous_cols].mean().to_numpy(copy=True)
            cov_mat = sub_df[continuous_cols].cov().to_numpy(copy=True)
            cov_mat += np.eye(len(continuous_cols)) * 1e-4  # Regularization
            
            synth_cont = np.random.multivariate_normal(mean_vec, cov_mat, size=n_needed)
            min_bounds = sub_df[continuous_cols].min().to_numpy()
            max_bounds = sub_df[continuous_cols].max().to_numpy()
            synth_cont = np.clip(synth_cont, min_bounds, max_bounds)
            
            synth_sub_df = pd.DataFrame(synth_cont, columns=continuous_cols)
        else:
            synth_sub_df = pd.DataFrame(index=range(n_needed))
            
        # Discrete features: sample from empirical conditional distribution
        for col in discrete_cols:
            val_counts = sub_df[col].value_counts(normalize=True)
            vals = val_counts.index.values
            probs = val_counts.values
            synth_sub_df[col] = np.random.choice(vals, size=n_needed, p=probs)
            
        synth_sub_df[target_col] = c
        synthetic_records.append(synth_sub_df)
        
    all_synth = pd.concat(synthetic_records, ignore_index=True)
    all_synth = all_synth[df.columns]
    
    for col in df.columns:
        if df[col].dtype == 'int64':
            all_synth[col] = all_synth[col].round().astype('int64')
            
    # Combine original clinical data + synthetic data
    mixed_df = pd.concat([df, all_synth], ignore_index=True).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return mixed_df

## 3. Generate Full Mixed Datasets & Export
- **Heart Disease**: Expanded from 302 unique to **2,000 balanced records** (1,000 Healthy, 1,000 Disease).
- **Parkinson's Disease**: Expanded from 195 to **1,200 balanced records** (600 Healthy, 600 Parkinson's).
- **Diabetes**: Expanded from 768 to **2,000 balanced records** (1,000 Non-diabetic, 1,000 Diabetic).

In [4]:
# Generate mixed datasets
mixed_heart = generate_synthetic_data(raw_heart, 'target', target_total_samples=2000, random_state=42)
mixed_park = generate_synthetic_data(raw_park, 'status', target_total_samples=1200, random_state=42)
mixed_diab = generate_synthetic_data(raw_diab, 'Outcome', target_total_samples=2000, random_state=42)

# Save to processed_data directory
mixed_heart.to_csv(os.path.join(OUTPUT_DIR, "cleaned_heart_disease.csv"), index=False)
mixed_park.to_csv(os.path.join(OUTPUT_DIR, "cleaned_parkinsons.csv"), index=False)
mixed_diab.to_csv(os.path.join(OUTPUT_DIR, "cleaned_diabetes.csv"), index=False)

print("[OK] Saved full mixed datasets to processed_data/")

[OK] Saved full mixed datasets to processed_data/


## 4. Verification Summary of Mixed Datasets

In [5]:
print("=" * 75)
print("AUGMENTED CLINICAL DATASET SUMMARY")
print("=" * 75)
for file, target in [("cleaned_heart_disease.csv", "target"),
                     ("cleaned_parkinsons.csv", "status"),
                     ("cleaned_diabetes.csv", "Outcome")]:
    df = pd.read_csv(os.path.join(OUTPUT_DIR, file))
    counts = df[target].value_counts().to_dict()
    print(f"{file:<28} | Rows: {df.shape[0]:<5} | Features: {df.shape[1]-1:<2} | Class Balance: {counts}")

AUGMENTED CLINICAL DATASET SUMMARY
cleaned_heart_disease.csv    | Rows: 2000  | Features: 13 | Class Balance: {1: 1000, 0: 1000}
cleaned_parkinsons.csv       | Rows: 1200  | Features: 22 | Class Balance: {0: 600, 1: 600}
cleaned_diabetes.csv         | Rows: 2000  | Features: 8  | Class Balance: {0: 1000, 1: 1000}
